## HW7 — EDA и отбор признаков Taxi

Добавьте календарные и бинарные признаки, исследуйте данные с помощью графиков, затем примените корреляционный анализ и прямой отбор пяти признаков. Основа — `EDA/HW_7.ipynb`; порядок ячеек и незавершённые упражнения сохранены. Полный CSV и старые графики заново не пересчитывались.

In [ ]:
import numpy as np
import pandas as pd

processed_data = pd.read_csv('processed_data1.csv', index_col='id')

processed_data.head()

In [ ]:
initial_data = pd.read_csv('taxi_data2.csv')

initial_data.head()

In [ ]:
initial_data.shape[0] == processed_data.shape[0]

In [ ]:
### Вернем в датасет колонку pickup_datetime

initial_data = initial_data.set_index('id')

processed_data = pd.merge(processed_data, initial_data['pickup_datetime'],
                          left_index=True, right_index=True)

In [ ]:
processed_data.head()

### Признаки времени

Верните время начала поездки по `id`. Из `pickup_datetime` выделите дату, день недели, час и месяц, чтобы исследовать изменения количества поездок и средней логарифмированной длительности.

In [ ]:
processed_data['pickup_datetime'] = pd.to_datetime(processed_data['pickup_datetime'])

processed_data['date'] = processed_data.pickup_datetime.dt.date
processed_data['day_of_week'] = processed_data.pickup_datetime.dt.dayofweek
processed_data['hour'] = processed_data.pickup_datetime.dt.hour
processed_data['month'] = processed_data.pickup_datetime.dt.month

In [ ]:
processed_data.head()

### Поездки по датам

Постройте графики числа поездок и среднего `log_trip_duration` по датам.

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()

fig.set_size_inches(16, 10)

ax_1 = fig.add_subplot(1, 2, 1)
plt.hist(processed_data['date'], bins=processed_data.date.unique().shape[0])
plt.ylim((4000, 10000))

ax_2 = fig.add_subplot(1, 2, 2)
plt.bar(sorted(list(processed_data['date'].unique())),
        processed_data.groupby('date', as_index=False)['log_trip_duration'].mean()['log_trip_duration'],
        width=1)
plt.ylim((6, 7))

fig.tight_layout()

ax_1.set(title = 'Количество поездок по датам')
ax_2.set(title = 'Средняя логарифмированная продолжительность поездок по датам')

plt.show()

### Исследование календарных зависимостей

Сравните число поездок и среднюю длительность по дням недели, часам и месяцам. Рассмотрите среднюю длительность по часам отдельно для дней недели, зависимости по месяцам и boxplot календарных групп. Для вопросов задания достаточно информативной части этих графиков.

Обратите внимание на январский провал числа поездок, который в задании связывается со снежной бурей. Выводы и численные значения по полным данным здесь не подтверждены.

In [ ]:
### Your code is here
# Index(['vendor_id', 'passenger_count', 'store_and_fwd_flag', 'distance_km',
#        'log_trip_duration', 'feature_1', 'feature_2', 'feature_3', 'feature_4',
#        'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9',
#        'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14',
#        'feature_15', 'feature_16', 'feature_17', 'feature_18', 'feature_19',
#        'feature_20', 'pickup_datetime', 'date', 'day_of_week', 'hour',
#        'month'],
#       dtype='str')
import seaborn as sns

# fig = plt.figure()
# fig.set_size_inches(16, 10)

plt.figure(figsize=(8, 4))
sns.countplot(data=processed_data, x='day_of_week')
plt.title('Количество поездок по дням недели')
plt.show()

# 2. График средней продолжительности (Seaborn сам посчитает среднее значение!)
plt.figure(figsize=(8, 4))
sns.barplot(data=processed_data, x='day_of_week', y='log_trip_duration')
plt.title('Средняя продолжительность (в log) по дням недели')
plt.show()

# sns.countplot(data=processed_data,
#               x='count trips by data',
#               y='day_of_week')

In [ ]:
hourly = processed_data.groupby('hour').agg(
    trips=('log_trip_duration', 'size'),
    avg_duration=('log_trip_duration', 'mean')
)

print(hourly.corr())

In [ ]:

plt.figure(figsize=(12, 8))
sns.boxplot(data=processed_data, x='day_of_week', y='log_trip_duration',palette= 'Set2', hue_order ='day_of_week')

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=processed_data,x = 'date', y = 'log_trip_duration',palette= 'Set2')

In [ ]:
monthly_weekday = (
    processed_data
    .groupby(['month', 'day_of_week'])['log_trip_duration']
    .mean()
    .unstack()
)

display(monthly_weekday.round(3))

plt.figure(figsize=(10, 5))

sns.heatmap(
    monthly_weekday,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
)

plt.xlabel('День недели')
plt.ylabel('Месяц')
plt.title('Средняя продолжительность поездок')
plt.show()

In [ ]:
sns.boxplot(
    data=processed_data,
    x='day_of_week',
    y='log_trip_duration'
)

In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=processed_data,x = 'month', y = 'log_trip_duration',palette= 'Set2')
# processed_data['month'].unique()

In [ ]:
plt.figure(figsize=(16, 8))
sns.boxplot(data=processed_data,x = 'hour', y = 'log_trip_duration',palette= 'Set2')

In [ ]:
plt.figure(figsize=(16, 8))
sns.barplot(data=processed_data,x=  'day_of_week', y= 'log_trip_duration',palette= 'Set2')

In [ ]:
processed_data.columns

### Два бинарных признака

`anomaly`: поездка пришлась на обнаруженный в задании день снежной бури (в разборе — 23 января 2016 года).

`traffic_jam`: для каждой пары «день недели — час» найдите средний `log_trip_duration`, выберите 10 наибольших средних и отметьте поездки, попавшие в эти пары. Здесь это упражнение EDA на всей таблице; оно не подтверждает независимую оценку качества на новых данных.

In [ ]:
### Создадим первый бинарный признак
### Your code is here
storm_date = pd.Timestamp('2016-01-23').date()
processed_data['anomaly'] = (processed_data['date'] == storm_date).astype(int)

In [ ]:
### Создадим второй бинарный признак
### Your code is here
# 1. Находим топ-10 пар (день-час) с самой большой средней продолжительностью
top_10 = processed_data.groupby(['day_of_week', 'hour'])['log_trip_duration'].mean().nlargest(10).index
# 2. Создаем пары из текущих строк и проверяем их присутствие в топ-10
processed_data['traffic_jam'] = processed_data.set_index(['day_of_week', 'hour']).index.isin(top_10).astype(int)

In [ ]:
processed_data.head()

In [ ]:
# processed_data['traffic_jam'].value_counts()

### Кодирование календарных признаков

Удалите `pickup_datetime` и `date`. Сохраните день недели, час и месяц как категориальные признаки и выполните One-Hot Encoding. Исходный цикл с `get_dummies` оставлен. Полное кодирование и кодирование с удалением одной категории допустимы; число колонок зависит также от состава исходной таблицы.

In [ ]:
processed_data = processed_data.drop(['pickup_datetime', 'date'], axis=1)

In [ ]:
### Делаем OneHotEncoding и конкатим с processed_data


for col in ['day_of_week', 'hour', 'month']:
     # 1. Создаем OHE-колонки для текущего признака
    dummies = pd.get_dummies(processed_data[col], prefix=col, dtype=int)

    # 2. Конкатенируем новые колонки и удаляем старую
    processed_data = pd.concat([processed_data.drop(columns=[col]), dummies], axis=1)

### От визуализации к признакам

В этом упражнении графики используются для выбора новых признаков ещё до обучения итоговой модели.

### Корреляционный анализ

Вычислите корреляцию `distance_km` и `passenger_count`. Есть ли по этой паре основание подозревать избыточность признаков? Численный ответ зависит от предоставленных данных.

In [ ]:
correlation = processed_data['distance_km'].corr(processed_data['passenger_count'])
correlation

In [ ]:
processed_data.head()

In [ ]:
### На данный момент у нас 41 фича
### Представим, что хочется сократить их количество до 5.
### Воспользуемся для этим каким-нибудь методом обертки
### Например, метод прямого отбора
from sklearn.linear_model import LinearRegression
### Your code is here
from sklearn.feature_selection import SequentialFeatureSelector

model = LinearRegression()
x = processed_data.drop('log_trip_duration', axis=1)
y = processed_data['log_trip_duration']

sfs = SequentialFeatureSelector(estimator=model, n_features_to_select=5)
sfs.fit(x, y)

In [ ]:
### Перечислите фичи, которые окажутся наиболее желанными
### Your code is here

In [ ]:
### Представим, что мы с Вами ничего не знаем про Кросс-Валидацию и отложенную выборку
### Замерьте качество линейной регрессии на данных фичах
### Сильно ли оно отличается от полученного на Кросс-Валидации в прошлом уроке в ДЗ?

### Your code is here